## Step 1: Read files from Volume

In [0]:
volume_path = "/Volumes/nyc_taxi/raw/taxi_files"
display(dbutils.fs.ls(volume_path))

path,name,size,modificationTime
dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-01.parquet,yellow_tripdata_2026-01.parquet,64165080,1782587160000
dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet,58683353,1782587156000
dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet,yellow_tripdata_2026-03.parquet,67891249,1782587161000


## Step 2: Read all parquet files

In [0]:
df_raw = spark.read.parquet("/Volumes/nyc_taxi/raw/taxi_files")

display(df_raw.limit(10))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2,2026-03-01T00:02:26.000,2026-03-01T00:13:45.000,1,2.58,1,N,48,151,1,14.2,1.0,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
2,2026-03-01T00:19:33.000,2026-03-01T00:28:21.000,1,1.5,1,N,238,166,1,10.0,1.0,0.5,0.0,0.0,1.0,15.0,2.5,0.0,0.0
2,2026-03-01T00:07:20.000,2026-03-01T00:15:12.000,2,0.88,1,N,90,249,1,8.6,1.0,0.5,2.87,0.0,1.0,17.22,2.5,0.0,0.75
2,2026-03-01T00:16:11.000,2026-03-01T00:28:20.000,1,1.76,1,N,249,137,1,12.1,1.0,0.5,3.57,0.0,1.0,21.42,2.5,0.0,0.75
2,2026-03-01T00:20:47.000,2026-03-01T00:30:44.000,2,1.57,1,N,100,142,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75
2,2026-03-01T00:53:06.000,2026-03-01T01:01:42.000,2,1.56,1,N,48,186,1,10.0,1.0,0.5,0.0,0.0,1.0,15.75,2.5,0.0,0.75
2,2026-02-28T23:58:32.000,2026-03-01T00:07:53.000,1,2.0,1,N,249,48,1,11.4,1.0,0.5,0.09,0.0,1.0,17.24,2.5,0.0,0.75
2,2026-03-01T00:30:09.000,2026-03-01T00:45:35.000,1,2.86,1,N,107,261,1,17.0,1.0,0.5,4.55,0.0,1.0,27.3,2.5,0.0,0.75
2,2026-03-01T00:46:47.000,2026-03-01T01:05:01.000,1,5.84,1,N,261,170,1,28.2,1.0,0.5,0.0,0.0,1.0,33.95,2.5,0.0,0.75
2,2026-03-01T00:05:51.000,2026-03-01T00:10:10.000,1,0.97,1,N,246,50,1,6.5,1.0,0.5,3.06,0.0,1.0,15.31,2.5,0.0,0.75


## Step 3: Check Schema

In [0]:
df_raw.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



## Step 4: Record Count

In [0]:
print(f"Total Records = {df_raw.count():,}")

Total Records = 11,077,206


## Step 5: Create Bronze Delta Table

In [0]:
(
    df_raw.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("nyc_taxi.bronze.taxi_trip_bronze")
)

## Step 6: Validate

In [0]:
%sql
select count(*) from nyc_taxi.bronze.taxi_trip_bronze

count(*)
11077206


## Step 7: Explore table

In [0]:
%sql
select * from nyc_taxi.bronze.taxi_trip_bronze limit 10

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2,2026-01-01T00:54:04.000,2026-01-01T00:59:37.000,1,0.97,1,N,239,238,1,7.2,1.0,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.0
1,2026-01-01T00:34:04.000,2026-01-01T00:39:47.000,0,0.9,1,N,163,162,2,7.9,4.25,0.5,0.0,0.0,1.0,13.65,2.5,0.0,0.75
1,2026-01-01T00:57:06.000,2026-01-01T01:05:59.000,0,1.4,1,N,43,237,1,10.7,4.25,0.5,2.5,0.0,1.0,18.95,2.5,0.0,0.75
2,2026-01-01T00:15:22.000,2026-01-01T00:58:10.000,4,5.58,1,N,142,209,1,38.7,1.0,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
2,2026-01-01T00:27:13.000,2026-01-01T00:40:43.000,0,2.16,1,N,88,144,1,13.5,1.0,0.5,3.85,0.0,1.0,23.1,2.5,0.0,0.75
2,2026-01-01T00:47:11.000,2026-01-01T01:00:47.000,2,2.33,1,N,144,137,1,14.2,1.0,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
1,2026-01-01T00:17:54.000,2026-01-01T00:28:32.000,1,1.3,1,N,142,50,2,11.4,4.25,0.5,0.0,0.0,1.0,17.15,2.5,0.0,0.75
1,2026-01-01T00:34:28.000,2026-01-01T00:59:05.000,0,2.9,1,N,50,234,1,22.6,4.25,0.5,5.65,0.0,1.0,34.0,2.5,0.0,0.75
2,2026-01-01T00:34:14.000,2026-01-01T01:11:58.000,1,5.34,1,N,161,45,1,37.3,1.0,0.5,8.61,0.0,1.0,51.66,2.5,0.0,0.75
2,2026-01-01T00:41:07.000,2026-01-01T00:50:42.000,3,1.83,1,N,237,263,1,10.7,1.0,0.5,2.36,0.0,1.0,18.06,2.5,0.0,0.0


In [0]:
df_raw = (
    spark.read
    .parquet("/Volumes/nyc_taxi/raw/taxi_files/*.parquet")
    .select("*", "_metadata")
)
display(df_raw.select("_metadata").limit(5))

_metadata
"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 0)"
"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 1)"
"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 2)"
"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 3)"
"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 4)"


In [0]:
from pyspark.sql.functions import current_timestamp,col

df_bronze = (
    df_raw
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file_path", df_raw["_metadata.file_path"])
    .withColumn("source_file_name",df_raw["_metadata.file_name"])
)
display(df_bronze.limit(10))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,_metadata,ingestion_timestamp,source_file_path,source_file_name
2,2026-03-01T00:02:26.000,2026-03-01T00:13:45.000,1,2.58,1,N,48,151,1,14.2,1.0,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75,"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 0)",2026-06-27T19:34:56.715Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet,yellow_tripdata_2026-03.parquet
2,2026-03-01T00:19:33.000,2026-03-01T00:28:21.000,1,1.5,1,N,238,166,1,10.0,1.0,0.5,0.0,0.0,1.0,15.0,2.5,0.0,0.0,"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 1)",2026-06-27T19:34:56.715Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet,yellow_tripdata_2026-03.parquet
2,2026-03-01T00:07:20.000,2026-03-01T00:15:12.000,2,0.88,1,N,90,249,1,8.6,1.0,0.5,2.87,0.0,1.0,17.22,2.5,0.0,0.75,"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 2)",2026-06-27T19:34:56.715Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet,yellow_tripdata_2026-03.parquet
2,2026-03-01T00:16:11.000,2026-03-01T00:28:20.000,1,1.76,1,N,249,137,1,12.1,1.0,0.5,3.57,0.0,1.0,21.42,2.5,0.0,0.75,"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 3)",2026-06-27T19:34:56.715Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet,yellow_tripdata_2026-03.parquet
2,2026-03-01T00:20:47.000,2026-03-01T00:30:44.000,2,1.57,1,N,100,142,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75,"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 4)",2026-06-27T19:34:56.715Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet,yellow_tripdata_2026-03.parquet
2,2026-03-01T00:53:06.000,2026-03-01T01:01:42.000,2,1.56,1,N,48,186,1,10.0,1.0,0.5,0.0,0.0,1.0,15.75,2.5,0.0,0.75,"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 5)",2026-06-27T19:34:56.715Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet,yellow_tripdata_2026-03.parquet
2,2026-02-28T23:58:32.000,2026-03-01T00:07:53.000,1,2.0,1,N,249,48,1,11.4,1.0,0.5,0.09,0.0,1.0,17.24,2.5,0.0,0.75,"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 6)",2026-06-27T19:34:56.715Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet,yellow_tripdata_2026-03.parquet
2,2026-03-01T00:30:09.000,2026-03-01T00:45:35.000,1,2.86,1,N,107,261,1,17.0,1.0,0.5,4.55,0.0,1.0,27.3,2.5,0.0,0.75,"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 7)",2026-06-27T19:34:56.715Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet,yellow_tripdata_2026-03.parquet
2,2026-03-01T00:46:47.000,2026-03-01T01:05:01.000,1,5.84,1,N,261,170,1,28.2,1.0,0.5,0.0,0.0,1.0,33.95,2.5,0.0,0.75,"List(dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet, yellow_tripdata_2026-03.parquet, 67891249, 0, 67891249, 2026-06-27T19:06:01.000Z, 8)",2026-06-27T19:34:56.715Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-03.parquet,yellow_tripdata_2026-03.parquet
2,2026-

In [0]:
%sql
DROP TABLE IF EXISTS nyc_taxi.bronze.taxi_trip_bronze;

In [0]:
(
    df_bronze
    .drop("_metadata")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi.bronze.taxi_trip_bronze")
)

In [0]:
%sql
select count(*) from nyc_taxi.bronze.taxi_trip_bronze

count(*)
11077206


In [0]:
%sql
select source_file_name, count(*) from nyc_taxi.bronze.taxi_trip_bronze
group by source_file_name

source_file_name,count(*)
yellow_tripdata_2026-03.parquet,3952451
yellow_tripdata_2026-01.parquet,3724889
yellow_tripdata_2026-02.parquet,3399866


In [0]:

%sql
DESCRIBE nyc_taxi.bronze.taxi_trip_bronze;

col_name,data_type,comment
VendorID,int,null
tpep_pickup_datetime,timestamp_ntz,null
tpep_dropoff_datetime,timestamp_ntz,null
passenger_count,bigint,null
trip_distance,double,null
RatecodeID,bigint,null
store_and_fwd_flag,string,null
PULocationID,int,null
DOLocationID,int,null
payment_type,bigint,null


## NULL ANALYSIS

In [0]:
%sql
SELECT
COUNT(*) total_records,
SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) passenger_nulls,
SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) fare_nulls,
SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) distance_nulls
FROM nyc_taxi.bronze.taxi_trip_bronze;

total_records,passenger_nulls,fare_nulls,distance_nulls
11077206,3057123,0,0


## Check Duplicate records

In [0]:
%sql
SELECT COUNT(*) total_rows,
COUNT(DISTINCT CONCAT(
VendorID,
tpep_pickup_datetime,
tpep_dropoff_datetime,
PULocationID,
DOLocationID
)) distinct_rows
FROM nyc_taxi.bronze.taxi_trip_bronze;

total_rows,distinct_rows
11077206,10998128
